# Importaciones

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import pipeline

# Carga de dataset

In [2]:
df = pd.read_parquet(r"C:\Users\samue\OneDrive - Universidad Externado de Colombia\Proyectos\Diagnóstico Psicológico\Repositorio\Datasets\RMHD_limpio.parquet")

# Configuración de GoEmotions

Se carga el modelo [GoEmotions](https://huggingface.co/SamLowe/roberta-base-go_emotions) y se establecen hiperparámetros

In [ ]:
MAX_CHARS = 2000         
DEFAULT_THRESHOLD = 0.3  

classifier = pipeline(
    task="text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None,           
    truncation=True,       
    max_length=512,
)

VALID_LABELS = {cfg for cfg in classifier.model.config.label2id.keys()}

Además, se establece el mapeo de las emociones hacia el modelo de Eckman

In [ ]:
EKMAN = {
    "anger":    ["anger", "annoyance", "disapproval"],
    "disgust":  ["disgust"],
    "fear":     ["fear", "nervousness"],
    "joy":      ["joy", "amusement", "approval", "excitement", "gratitude", "love",
                 "optimism", "relief", "pride", "admiration", "desire", "caring"],
    "sadness":  ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral":  ["neutral"],
}
COLS_EMO = list(EKMAN.keys())

Se establecen funciones para realizar la predicción

In [ ]:
def predecir_emociones(textos, umbral=DEFAULT_THRESHOLD, top_n=3):
    """
    Predice emociones para uno o varios textos.

    Args:
        textos: str o lista de str.
        umbral: score mínimo (0-1) para considerar una emoción activa.
        top_n: cuántas emociones principales devolver.

    Returns:
        Lista de dicts con: texto, emociones_activas, top, scores (todas).
    """
    if isinstance(textos, str):
        textos = [textos]
    if not isinstance(textos, (list, tuple)) or len(textos) == 0:
        raise ValueError("Debes pasar un texto o una lista no vacía de textos.")

    salidas = classifier(textos)  # lista de listas de {label, score}

    resultados = []
    for original, salida in zip(textos, salidas):
        scores = {d["label"]: d["score"] for d in salida}
        # Validación de salida: etiquetas esperadas y scores en [0,1]
        assert set(scores) == VALID_LABELS, "Etiquetas inesperadas en la salida del modelo."
        assert all(0.0 <= s <= 1.0 for s in scores.values()), "Score fuera de rango."

        ordenadas = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        activas = [(l, round(s, 4)) for l, s in ordenadas if s >= umbral]
        resultados.append({
            "texto": original,
            "emociones_activas": activas,
            "top": [(l, round(s, 4)) for l, s in ordenadas[:top_n]],
            "scores": scores,
        })
    return resultados


def mostrar(resultados):
    for r in resultados:
        print(f"📝 {r['texto'][:100]}{'...' if len(r['texto']) > 100 else ''}")
        print(f"   Top: {r['top']}")
        activas = r["emociones_activas"]
        print(f"   Activas (≥ umbral): {activas if activas else 'ninguna'}\n")

# Predicciones

Se realiza la predicción sobre el texto limpio

In [ ]:
def limpiar(t):
    if not isinstance(t, str):
        return ""
    return " ".join(t.split())[:MAX_CHARS]

textos = df["text"].map(limpiar)

salidas = classifier(textos.tolist(), batch_size=32)

filas = []
for salida in salidas:
    p = {d["label"]: d["score"] for d in salida}
    filas.append({emo: max(p[l] for l in labels) for emo, labels in EKMAN.items()})

res = pd.DataFrame(filas, index=df.index, columns=COLS_EMO)

Se unifica el dataframe con las emociones

In [ ]:
df = df.drop(columns=COLS_EMO + ["emocion"], errors="ignore")
df = df.join(res)
df["emocion"] = df[COLS_EMO].idxmax(axis=1)

print("Distribución de emociones predichas:")
print(df["emocion"].value_counts(normalize=True).round(3))

df = df[["subreddit", "year", "month", "text", "text_clean"] + COLS_EMO + ["emocion"]]
df.head()